# Midnight-aligned snap test

**Date:** 2026-04-18.
**Hypothesis (Jake):** the h/m under-prediction is partly an artifact of snap-time semantics. Current convention `snap_time = close_ts - 3d = 14:00 UTC on close-3` creates an asymmetry:

- **Day-level training movies:** their reviews on day close-3 are timestamped at midnight UTC of close-3. In dbc terms (from close_ts=14:00 UTC), that's `dbc = 3.583` → OBSERVED (out of phase_1). KDE training has no mass in the `(2.58, 3.0)` dbc zone.
- **H/m targets:** reviews on day close-3 happen at real times. A review at 18:00 UTC on close-3 has `dbc = 2.833` → in phase_1. This is a zone the KDE is blind to (no training data there).

Proposed fix: midnight-align the snap. `snap_time = close_ts.floor('D') - 3d = midnight UTC on close-3`. Under this:

- **Day-level training:** reviews on close-3 (dbc=3.583) fall AT snap boundary — cleanly in phase_1 or out.
- **H/m targets:** same day-boundary semantics as training.
- **No blind zone** between training and target.

**Test:** rerun weighted-KDE prediction with midnight-aligned snap. Compare h/m MAE vs original snap convention.

**Expected outcome if hypothesis correct:** h/m under-prediction shrinks because predictions now capture the full 3 calendar days of reviews (instead of 2.417d) and the zone asymmetry is eliminated.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP_DAYS = 3  # "T-3d"

CACHE = CACHE_DIR / 'midnight_snap_test.pkl'
print('Ready.')

## Redefined snap logic

Under midnight-aligned convention:
- `snap_time = close_ts.floor('D') - SNAP_DAYS days`
- `snap_dbc_effective = (close_ts - snap_time) / 86400 = SNAP_DAYS + midnight_utc_dbc`
- `midnight_utc_dbc` (upper phase-2 boundary) is unchanged at `(close_ts - close_ts.floor('D'))/86400`
- Phase-1 integration window in dbc: `(midnight_utc_dbc, snap_dbc_effective]`

In [ ]:
def midnight_snap_time(close_ts, snap_days):
    return close_ts.floor('D') - pd.Timedelta(days=snap_days)

def snapshot_state_midnight(target_slug, snap_time, target_close):
    obs = reviews[
        (reviews['movie_slug'] == target_slug)
        & (reviews['estimated_timestamp'] < snap_time)
        & (reviews['estimated_timestamp'] < target_close)
    ]
    if obs.empty:
        return None
    return {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float(
            (target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400
        ),
    }

def actual_phase1_midnight(target_slug, snap_time, target_close):
    """Count reviews with ts in [snap_time, midnight UTC of close_day)."""
    close_midnight = target_close.floor('D')
    mr = reviews[reviews['movie_slug'] == target_slug]
    in_window = (
        (mr['estimated_timestamp'] >= snap_time)
        & (mr['estimated_timestamp'] < close_midnight)
    )
    return int(in_window.sum())

def passes_skip_rules_midnight(state, snap_dbc_effective, min_critics=3):
    if state is None:
        return False
    # require at least 1d of observed window
    if state['first_review_dbc'] < snap_dbc_effective + 1.0:
        return False
    if len(state['observed_critics']) < min_critics:
        return False
    return True

# Sanity check on the_drama
sample = 'the_drama'
close_ts_s = close_date_map[sample]
snap_time_s = midnight_snap_time(close_ts_s, SNAP_DAYS)
snap_dbc_eff_s = (close_ts_s - snap_time_s).total_seconds() / 86400
midnight_utc_dbc_s = (close_ts_s - close_ts_s.floor('D')).total_seconds() / 86400
state_s = snapshot_state_midnight(sample, snap_time_s, close_ts_s)
actual_p1_s = actual_phase1_midnight(sample, snap_time_s, close_ts_s)
print(f'{sample}:')
print(f'  close_ts: {close_ts_s}')
print(f'  snap_time (midnight-aligned): {snap_time_s}')
print(f'  snap_dbc_effective = {snap_dbc_eff_s:.3f}')
print(f'  midnight_utc_dbc = {midnight_utc_dbc_s:.3f}')
print(f'  phase_1 window = ({midnight_utc_dbc_s:.3f}, {snap_dbc_eff_s:.3f}]')
print(f'  observed_count: {state_s["observed_count"]}')
print(f'  first_review_dbc: {state_s["first_review_dbc"]:.3f}')
print(f'  actual_phase1 (midnight-aligned): {actual_p1_s}')
print()
# Compare to original snap
orig_snap = close_ts_s - pd.Timedelta(days=3)
orig_in_window = (
    (reviews[reviews['movie_slug']==sample]['estimated_timestamp'] > orig_snap.floor('D')) if False else
    None
)
print(f'  original snap_time: {close_ts_s - pd.Timedelta(days=3)}')
print(f'  (for reference, previous actual_phase1 was 52)')

## LOO sweep at T-3d midnight-aligned

In [ ]:
def run_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    skipped = {'low_first_review': 0, 'low_critics': 0, 'no_obs': 0, 'low_scores': 0}
    for target in close_date_map:
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = midnight_snap_time(target_close, SNAP_DAYS)
        snap_dbc_effective = (target_close - snap_time).total_seconds() / 86400

        state = snapshot_state_midnight(target, snap_time, target_close)
        if state is None:
            skipped['no_obs'] += 1
            continue
        if state['first_review_dbc'] < snap_dbc_effective + 1.0:
            skipped['low_first_review'] += 1
            continue
        if len(state['observed_critics']) < 3:
            skipped['low_critics'] += 1
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc_effective
        target_critics = state['observed_critics']

        scores = combined_score_with_scores(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
        )
        if len(scores) < 5:
            skipped['low_scores'] += 1
            continue

        try:
            profiles = build_weighted_critic_profiles(reviews, close_date_map, scores, verbose=False)
            if len(profiles.df) == 0:
                continue
            model = build_weighted_kde_lambda_model(
                profiles, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
            )
            pred = predict_window_custom(
                model, dbc_from=snap_dbc_effective, dbc_to=midnight_utc_dbc,
                observed_critics=target_critics,
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
        except Exception as e:
            continue

        actual_p1 = actual_phase1_midnight(target, snap_time, target_close)

        rows.append({
            'target': target,
            'target_gap': target_gap,
            'first_review_dbc': state['first_review_dbc'],
            'observed_count': state['observed_count'],
            'snap_dbc_effective': snap_dbc_effective,
            'pred_phase1': float(pred),
            'actual_phase1': actual_p1,
            'err': float(pred) - actual_p1,
        })

    print(f'Skipped: {skipped}')
    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_sweep()
print(f'\nn={len(results)}')

## Compare h/m subset MAE: midnight-aligned vs original

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

results['abs_err'] = results['err'].abs()
hm = results[results['target'].isin(HM)].copy()

print('Midnight-aligned snap results:')
cols = ['target', 'snap_dbc_effective', 'observed_count', 'actual_phase1', 'pred_phase1', 'err']
print(hm[cols].to_string(index=False, float_format='%.2f'))
print()
print(f'H/m MAE (midnight-aligned): {hm["abs_err"].mean():.2f}')
print(f'H/m mean_err:              {hm["err"].mean():+.2f}')
print()
print('For comparison, original snap (from prior experiments):')
print('  H/m MAE: 14.75, mean_err: -14.75')
print('  Per-target errors:')
print('    forbidden_fruits_2026:  -3.09 (actual 17, pred 13.91)')
print('    the_drama:             -33.67 (actual 52, pred 18.33)')
print('    super_mario:            -5.34 (actual 26, pred 20.66)')
print('    they_will_kill_you:    -21.59 (actual 34, pred 12.41)')
print('    you_me_and_tuscany:    -10.07 (actual 26, pred 15.93)')

## Full cohort comparison

In [ ]:
print(f'Midnight-aligned cohort (n={len(results)}):')
print(f'  MAE:      {results["abs_err"].mean():.2f}')
print(f'  mean_err: {results["err"].mean():+.2f}')
print()

# Stratified by actual_phase1 quartile
results['q_actual'] = pd.qcut(results['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
for q in ['Q1','Q2','Q3','Q4']:
    sub = results[results['q_actual'] == q]
    if not len(sub):
        continue
    lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
    print(f'{q} (actual [{lo}, {hi}]) n={len(sub)}:  MAE={sub["abs_err"].mean():.2f}  mean_err={sub["err"].mean():+.2f}')


## Diagnosis — did the asymmetry explain the under-prediction?

Key question: did `the_drama` MAE drop from ~33 toward something closer to cohort-typical? Did `they_will_kill_you` move?

If h/m MAE stays large (e.g., >10 per target), the asymmetry explains only a small part and the primary driver is still volume/cohort-coverage.